# Lab 7 Report: Robot Perception and Sensing - Alan Nur

## 1. License of yolo_ros
The `yolo_ros` package used in this lab is licensed under the GPL-3.0 (GNU General Public License v3.0). This license allows for free use, sharing, and modification, as long as any derivative work is also kept open source.

## 2. Camera Specifications
The OAK-D camera used on the Turtlebot4 provides stereo depth data. The effective stereo depth range is 0.7 meters to 12 meters. Objects within this range allow for accurate spatial identification by the YOLO model.

## 3. RViz Setup and Visualization
Based on the simulation runs on the ASUS G14 laptop, the RViz setup includes:
- **Robot Visualization**: The Turtlebot4 model is correctly placed within the warehouse world.
- **Camera Feed**: A live image stream from the OAK-D sensor is displayed in the Image viewer.
- **Center Point Marker**: A red sphere marker is generated at the center of the detected object to verify the distance calculation.
- **TF Tree**: The system successfully tracks the transformation from `oakd_rgb_camera_optical_frame` to `base_link`.

## 4. Distance Measurements (Base Link Frame)
The following data shows the distance of a person identified in the simulation. These values represent the position of the object relative to the center of the robot.

| Trial | Detected Object | X-axis (m) | Y-axis (m) | Z-axis (m) | Total Distance (m) |
|---|---|---|---|---|---|
| 1 | Person (Close) | 1.92 | 0.11 | 0.89 | 2.12 |
| 2 | Person (Mid) | 2.35 | 1.40 | 1.04 | 2.92 |
| 3 | Person (Mid) | 2.49 | 1.25 | 1.09 | 2.99 |
| 4 | Person (Far) | 3.04 | 1.42 | 1.26 | 3.58 |
| 5 | Person (Wall) | 14.13 | 8.60 | 3.93 | 16.99 |

*Note: Total distance is calculated using the formula $d = \sqrt{x^2 + y^2 + z^2}$.*

## 5. Code Implementation
The following Python script was used to find the center point of the YOLO bounding box and transform the depth data into the `base_link` frame using `tf2_ros`.

In [ ]:
import rclpy
from rclpy.node import Node
from yolo_msgs.msg import DetectionArray
from sensor_msgs.msg import Image
from visualization_msgs.msg import Marker
from cv_bridge import CvBridge
import tf2_ros
from geometry_msgs.msg import PointStamped
from tf2_geometry_msgs import do_transform_point

class CenterPointNode(Node):
    def __init__(self):
        super().__init__('center_point_node')
        self.tf_buffer = tf2_ros.Buffer()
        self.tf_listener = tf2_ros.TransformListener(self.tf_buffer, self)
        self.det_sub = self.create_subscription(DetectionArray, '/yolo/detections', self.det_callback, 10)
        self.depth_sub = self.create_subscription(Image, '/oakd/rgb/preview/depth', self.depth_callback, 10)
        self.marker_pub = self.create_publisher(Marker, '/lab7/center_marker', 10)
        self.bridge = CvBridge()
        self.latest_depth_image = None

    def depth_callback(self, msg):
        self.latest_depth_image = self.bridge.imgmsg_to_cv2(msg, desired_encoding='passthrough')

    def det_callback(self, msg):
        if self.latest_depth_image is None or not msg.detections:
            return
        for det in msg.detections:
            cx = int(det.bbox.center.position.x)
            cy = int(det.bbox.center.position.y)
            z_meters = float(self.latest_depth_image[cy, cx])
            if z_meters > 0:
                cam_x = (cx - 320) * z_meters / 500.0
                cam_y = (cy - 240) * z_meters / 500.0
                self.transform_to_base_link(cam_x, cam_y, z_meters, det.class_name)

    def transform_to_base_link(self, x, y, z, name):
        try:
            t = self.tf_buffer.lookup_transform('base_link', 'oakd_rgb_camera_optical_frame', rclpy.time.Time())
            p_cam = PointStamped()
            p_cam.header.frame_id = 'oakd_rgb_camera_optical_frame'
            p_cam.point.x, p_cam.point.y, p_cam.point.z = x, y, z
            p_base = do_transform_point(p_cam, t)
            self.publish_marker(p_base.point.x, p_base.point.y, p_base.point.z, name)
        except tf2_ros.LookupException:
            pass

    def publish_marker(self, x, y, z, name):
        marker = Marker()
        marker.header.frame_id = 'base_link'
        marker.type = Marker.SPHERE
        marker.pose.position.x, marker.pose.position.y, marker.pose.position.z = x, y, z
        marker.scale.x = marker.scale.y = marker.scale.z = 0.15
        marker.color.r, marker.color.a = 1.0, 1.0
        self.marker_pub.publish(marker)

def main(args=None):
    rclpy.init(args=args)
    rclpy.spin(CenterPointNode())
    rclpy.shutdown()

if __name__ == '__main__':
    main()

## 6. Video Demonstration

<iframe width="560" height="315" src="https://www.youtube.com/embed/Fh8GYgYhGa0" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" allowfullscreen></iframe>

**Source Link:** [https://youtu.be/Fh8GYgYhGa0](https://youtu.be/Fh8GYgYhGa0)